# 06 — Batched Primitives on Flash Attention KV Caches

This notebook validates that gather, scoring, selection, and scatter
work for a batch of sequences — following the total replacement
compression pipeline: **gather → score → select → scatter back**.

Gather operates on a flat `slot_mapping` that doesn't care about
sequence boundaries — batching is just concatenating slot mappings.
Scoring and selection require per-sequence semantics, so we pad to
`max_seq_len`, stack into kvpress's layout
`[batch, heads, seq_len, dim]`, and use `valid_mask` to exclude
padding. We call kvpress's `KeyDiffPress.score()` directly for
scoring, and replicate `ScorerPress.compress()`'s exact selection
logic (`topk` + `gather`) — both fully batched. Scatter writes the
compressed results back to the paged cache in one call.

We create three sequences with different lengths and validate:
1. **Batched gather** — concatenate slot mappings, gather in one call,
   split by sequence lengths
2. **Batched scoring** — pad + mask, score with kvpress, verify against
   per-sequence scoring
3. **Batched selection** — `topk` + `gather` on padded tensors, verify
   against per-sequence selection
4. **Batched scatter** — write compressed keys/values back to cache,
   verify round trip

## Imports and Setup

In [1]:
import torch
from torch.nn.utils.rnn import pad_sequence
from vllm import _custom_ops as ops
from kvpress import KeyDiffPress

torch.manual_seed(42)
torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA A100-SXM4-40GB


## Configuration

In [2]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
DEVICE = "cuda"

SEQ_LENS = [137, 64, 200]
COMPRESSION_RATIO = 0.5

## Helper Functions

In [3]:
def create_kv_caches_flash(num_blocks, block_size, num_kv_heads, head_size,
                           dtype, device="cuda"):
    key_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    return key_cache, value_cache


def build_slot_mapping_for_positions(block_table, positions, block_size):
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping, block_size):
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size
    keys = key_cache[block_indices, offsets]
    values = value_cache[block_indices, offsets]
    return keys, values   

print("Helper functions defined")

Helper functions defined


## Populate Cache

Create three sequences with different lengths, assign each a disjoint
range of physical blocks, and scatter them into the paged cache.
This is setup — single-sequence scatter is validated in prior notebooks.

In [4]:
total_blocks = sum((s + BLOCK_SIZE - 1) // BLOCK_SIZE for s in SEQ_LENS)
key_cache, value_cache = create_kv_caches_flash(
    total_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

seq_data = []
block_offset = 0
for seq_len in SEQ_LENS:
    num_seq_blocks = (seq_len + BLOCK_SIZE - 1) // BLOCK_SIZE
    block_table = torch.arange(
        block_offset, block_offset + num_seq_blocks,
        dtype=torch.long, device=DEVICE,
    )
    positions = torch.arange(seq_len, dtype=torch.long, device=DEVICE)
    slot_mapping = build_slot_mapping_for_positions(block_table, positions, BLOCK_SIZE)

    keys = torch.randn(seq_len, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)
    values = torch.randn(seq_len, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)

    k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
    v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
    ops.reshape_and_cache_flash(
        keys, values, key_cache, value_cache,
        slot_mapping, "auto", k_scale, v_scale,
    )

    seq_data.append((keys, values, slot_mapping, block_table))
    block_offset += num_seq_blocks

print(f"Populated cache with {len(SEQ_LENS)} sequences ({SEQ_LENS})")
print(f"Total blocks: {total_blocks}")

Populated cache with 3 sequences ([137, 64, 200])
Total blocks: 26


## Batched Gather

Concatenate all slot mappings, gather in one call, then
`torch.split` by sequence lengths to recover each sequence's tensors.

In [5]:
all_slots_cat = torch.cat([sd[2] for sd in seq_data])
keys_gathered, values_gathered = gather_from_paged_cache(
    key_cache, value_cache, all_slots_cat, BLOCK_SIZE,
)

keys_per_seq = torch.split(keys_gathered, SEQ_LENS)
values_per_seq = torch.split(values_gathered, SEQ_LENS)

for i, (keys_orig, values_orig, _, _) in enumerate(seq_data):
    torch.testing.assert_close(keys_per_seq[i], keys_orig, atol=0, rtol=0)
    torch.testing.assert_close(values_per_seq[i], values_orig, atol=0, rtol=0)

print(f"All {len(SEQ_LENS)} sequences verified — batched gather is correct")

All 3 sequences verified — batched gather is correct


In [6]:
import torch
from torch import nn
from torch.nn import functional as F

from kvpress.presses.scorer_press import ScorerPress

def my_score(
        module: nn.Module,
        hidden_states: torch.Tensor,
        keys: torch.Tensor,
        values: torch.Tensor,
        attentions: torch.Tensor,
        kwargs,
    ) -> torch.Tensor:
        valid_mask = kwargs.get("valid_mask") if isinstance(kwargs, dict) else None
        normalized = F.normalize(keys, p=2, dim=-1)

        if valid_mask is not None:
            mask = valid_mask.unsqueeze(-1)
            anchor = normalized.masked_fill(~mask, 0).sum(dim=2, keepdim=True) / mask.sum(dim=2, keepdim=True).clamp(min=1)
        else:
            anchor = normalized.mean(dim=2, keepdim=True)

        return -F.cosine_similarity(keys, anchor, dim=-1)

## Batched Scoring

After batched gather we have per-sequence dense tensors of different
lengths. To score them in a single kvpress call we:
1. Permute each to `[heads, seq_len, dim]` and pad to `max_seq_len`
2. Stack into `[batch, heads, max_seq_len, dim]`
3. Build a `valid_mask` of shape `[batch, 1, max_seq_len]`
4. Call `KeyDiffPress.score()` with the mask

We verify against scoring each sequence individually (no padding).

In [7]:
max_seq_len = max(SEQ_LENS)
press = KeyDiffPress()

keys_by_head = [k.permute(1, 0, 2) for k in keys_per_seq]
keys_padded = pad_sequence(
    [k.transpose(0, 1) for k in keys_by_head],
    batch_first=True,
).transpose(1, 2)

values_by_head = [v.permute(1, 0, 2) for v in values_per_seq]
values_padded = pad_sequence(
    [v.transpose(0, 1) for v in values_by_head],
    batch_first=True,
).transpose(1, 2)

valid_mask = torch.zeros(len(SEQ_LENS), 1, max_seq_len, dtype=torch.bool, device=DEVICE)
for i, sl in enumerate(SEQ_LENS):
    valid_mask[i, :, :sl] = True

batched_scores = my_score(
    module=None, hidden_states=None,
    keys=keys_padded, values=None, attentions=None,
    kwargs={"valid_mask": valid_mask},
)

batched_scores[~valid_mask.expand_as(batched_scores)] = float("-inf")

per_seq_scores = []
for k in keys_per_seq:
    k_single = k.permute(1, 0, 2).unsqueeze(0)
    s = press.score(
        module=None, hidden_states=None,
        keys=k_single, values=None, attentions=None,
        kwargs={},
    )
    per_seq_scores.append(s.squeeze(0))

for i, sl in enumerate(SEQ_LENS):
    torch.testing.assert_close(
        batched_scores[i, :, :sl], per_seq_scores[i],
        atol=1e-3, rtol=1e-3,
    )

print(f"All {len(SEQ_LENS)} sequences verified — batched scoring matches per-sequence scoring")
print(f"Padded keys:   {keys_padded.shape}")
print(f"Padded values: {values_padded.shape}")
print(f"Scores:        {batched_scores.shape}")

All 3 sequences verified — batched scoring matches per-sequence scoring
Padded keys:   torch.Size([3, 4, 200, 128])
Padded values: torch.Size([3, 4, 200, 128])
Scores:        torch.Size([3, 4, 200])


## Batched Selection

This is `ScorerPress.compress()`'s exact selection logic applied to our
padded tensors. `topk` and `gather` on `[batch, heads, seq_len, dim]`
naturally parallelize across both batch and heads dimensions. Padding
scores are already set to `-inf` so they are never selected.

In [8]:
k_len = keys_padded.shape[2]
n_kept = int(k_len * (1 - COMPRESSION_RATIO))
indices = batched_scores.topk(n_kept, dim=-1).indices
indices_expanded = indices.unsqueeze(-1).expand(-1, -1, -1, HEAD_SIZE)

keys_compressed = keys_padded.gather(2, indices_expanded).contiguous()
values_compressed = values_padded.gather(2, indices_expanded).contiguous()

per_seq_keys = []
per_seq_values = []
for i, sl in enumerate(SEQ_LENS):
    s = per_seq_scores[i].unsqueeze(0)
    k = keys_padded[i:i+1, :, :sl, :]
    v = values_padded[i:i+1, :, :sl, :]
    n = int(sl * (1 - COMPRESSION_RATIO))
    idx = s[:, :, :sl].topk(n, dim=-1).indices
    idx_exp = idx.unsqueeze(-1).expand(-1, -1, -1, HEAD_SIZE)
    per_seq_keys.append(k.gather(2, idx_exp).squeeze(0))
    per_seq_values.append(v.gather(2, idx_exp).squeeze(0))

for i, sl in enumerate(SEQ_LENS):
    n = int(sl * (1 - COMPRESSION_RATIO))
    batch_k = keys_compressed[i, :, :n, :]
    batch_v = values_compressed[i, :, :n, :]
    batch_k_sorted, _ = batch_k.sort(dim=1)
    per_k_sorted, _ = per_seq_keys[i].sort(dim=1)
    batch_v_sorted, _ = batch_v.sort(dim=1)
    per_v_sorted, _ = per_seq_values[i].sort(dim=1)
    torch.testing.assert_close(batch_k_sorted, per_k_sorted, atol=1e-4, rtol=1e-4)
    torch.testing.assert_close(batch_v_sorted, per_v_sorted, atol=1e-4, rtol=1e-4)

print(f"All {len(SEQ_LENS)} sequences verified — batched selection matches per-sequence selection")
print(f"Compressed keys:   {keys_compressed.shape}")
print(f"Compressed values: {values_compressed.shape}")
for i, sl in enumerate(SEQ_LENS):
    n = int(sl * (1 - COMPRESSION_RATIO))
    print(f"  Seq {i}: {sl} -> {n} tokens kept (first {n} of {n_kept} are valid)")

All 3 sequences verified — batched selection matches per-sequence selection
Compressed keys:   torch.Size([3, 4, 100, 128])
Compressed values: torch.Size([3, 4, 100, 128])
  Seq 0: 137 -> 68 tokens kept (first 68 of 100 are valid)
  Seq 1: 64 -> 32 tokens kept (first 32 of 100 are valid)
  Seq 2: 200 -> 100 tokens kept (first 100 of 100 are valid)


## Batched Scatter

Write the compressed keys/values back to the paged cache. For each
sequence, the compressed tokens go into the first `n_kept` slots.
We concatenate all sequences' compressed data and slot mappings,
scatter in one call, then gather back to verify.

In [9]:
all_compressed_keys = []
all_compressed_values = []
all_compressed_slots = []
compressed_lens = []

for i, sl in enumerate(SEQ_LENS):
    n = int(sl * (1 - COMPRESSION_RATIO))
    compressed_lens.append(n)

    k = keys_compressed[i, :, :n, :].permute(1, 0, 2).contiguous()
    v = values_compressed[i, :, :n, :].permute(1, 0, 2).contiguous()
    all_compressed_keys.append(k)
    all_compressed_values.append(v)

    _, _, _, block_table = seq_data[i]
    compact_positions = torch.arange(n, dtype=torch.long, device=DEVICE)
    compact_slots = build_slot_mapping_for_positions(block_table, compact_positions, BLOCK_SIZE)
    all_compressed_slots.append(compact_slots)

k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

ops.reshape_and_cache_flash(
    torch.cat(all_compressed_keys),
    torch.cat(all_compressed_values),
    key_cache, value_cache,
    torch.cat(all_compressed_slots),
    "auto", k_scale, v_scale,
)

for i in range(len(SEQ_LENS)):
    k_back, v_back = gather_from_paged_cache(
        key_cache, value_cache, all_compressed_slots[i], BLOCK_SIZE,
    )
    torch.testing.assert_close(k_back, all_compressed_keys[i], atol=0, rtol=0)
    torch.testing.assert_close(v_back, all_compressed_values[i], atol=0, rtol=0)

print(f"All {len(SEQ_LENS)} sequences verified — batched scatter of compressed data is correct")
for i, sl in enumerate(SEQ_LENS):
    print(f"  Seq {i}: {sl} -> {compressed_lens[i]} tokens written back")

All 3 sequences verified — batched scatter of compressed data is correct
  Seq 0: 137 -> 68 tokens written back
  Seq 1: 64 -> 32 tokens written back
  Seq 2: 200 -> 100 tokens written back


## Notes

All four primitives work in batched mode, following the total
replacement pipeline: **gather → score → select → scatter back**.

- **Gather** — concatenate slot mappings, one call, split by lengths
- **Scoring** — pad + `valid_mask`, call kvpress's `KeyDiffPress.score()`
- **Selection** — `ScorerPress.compress()`'s exact `topk` + `gather` on
  padded tensors, padding scores set to `-inf`
- **Scatter** — concatenate compressed data and slot mappings, one call

The full pipeline operates on a batch of sequences with different
lengths, using kvpress's own code for the compute-heavy steps.